# 04 — Validate, train, evaluate, and predict with the U-Net baseline

This notebook is the final guided stage. It validates training readiness, runs a synthetic overfit smoke test, optionally trains the real binary U-Net, evaluates a checkpoint on aligned human annotations, and predicts complete images.

The smoke test is always safe: it uses a generated 32×32 example and proves that model construction, loss, backpropagation, and optimization work. Real training remains disabled until you explicitly enable it and provide human-validated `seed`, `corrected`, or `reviewed` annotations.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

## User settings

By default this notebook runs only the synthetic smoke test. After importing human-validated masks and assigning independent splits, set `RUN_REAL_TRAINING=True`. The notebook writes a runtime copy of the configuration under `outputs/notebook_run/`; it does not rewrite the tracked YAML file.

In [ ]:
BASE_CONFIG_PATH = Path('configs/train_binary.yaml')
MANIFEST_OVERRIDE = None  # Example: Path('data/metadata/manifest_corrected.csv')
OUTPUT_DIRECTORY = Path('outputs/checkpoints/notebook_binary_baseline')

RUN_SMOKE_TEST = True
SMOKE_STEPS = 25
RUN_REAL_TRAINING = False
RUN_EVALUATION = False
RUN_PREDICTION = False
EVALUATION_SPLIT = 'val'
PREDICTION_SPLIT = 'test'

# Optional runtime overrides; None keeps the YAML value.
EPOCH_OVERRIDE = None
BATCH_SIZE_OVERRIDE = None
ENABLE_CROSS_VALIDATION = False
CV_SPLITS = 5
CV_VALIDATION_FOLD = 0
CV_GROUP_COLUMN = 'image_id'

## Create the runtime configuration

The binary model receives three planes: normalized GFAP, binary nucleus mask, and nucleus-proximity map. Its target remains background versus GFAP-positive astrocyte structure.

In [ ]:
import copy
import yaml

with BASE_CONFIG_PATH.open('r', encoding='utf-8') as handle:
    configuration = yaml.safe_load(handle)
configuration = copy.deepcopy(configuration)
if MANIFEST_OVERRIDE is not None:
    configuration['data']['manifest_path'] = str(Path(MANIFEST_OVERRIDE))
configuration['output']['directory'] = str(OUTPUT_DIRECTORY)
if EPOCH_OVERRIDE is not None:
    configuration['training']['epochs'] = int(EPOCH_OVERRIDE)
if BATCH_SIZE_OVERRIDE is not None:
    configuration['training']['batch_size'] = int(BATCH_SIZE_OVERRIDE)
if ENABLE_CROSS_VALIDATION:
    configuration['cross_validation'].update({
        'enabled': True,
        'n_splits': int(CV_SPLITS),
        'validation_fold': int(CV_VALIDATION_FOLD),
        'group_column': CV_GROUP_COLUMN,
    })

RUNTIME_CONFIG_PATH = Path('outputs/notebook_run/train_binary.yaml')
RUNTIME_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
with RUNTIME_CONFIG_PATH.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(configuration, handle, sort_keys=False)
print('Runtime configuration:', RUNTIME_CONFIG_PATH)
print(yaml.safe_dump(configuration, sort_keys=False))

## Training-readiness audit

Normal training needs at least one human-validated `train` image and one independent human-validated `val` image. Grouped cross-validation instead needs at least `n_splits` independent image/well groups. Patches never define folds; every patch inherits its source image's group.

In [ ]:
import pandas as pd

from astroseg.constants import TRAINABLE_ANNOTATION_STATUSES
from astroseg.io import load_manifest

active_manifest_path = Path(configuration['data']['manifest_path'])
manifest = load_manifest(active_manifest_path)
trainable = manifest['annotation_status'].isin(TRAINABLE_ANNOTATION_STATUSES)
audit = manifest.assign(trainable=trainable)[
    ['image_id', 'annotation_status', 'review_status', 'split', 'trainable', 'annotation_path']
]
print('Annotation states:')
print(manifest['annotation_status'].value_counts(dropna=False).to_string())
print()
print('Trainable rows by split:')
print(manifest.loc[trainable, 'split'].value_counts(dropna=False).to_string())
audit

In [ ]:
cross_validation_enabled = bool(configuration.get('cross_validation', {}).get('enabled', False))
if cross_validation_enabled:
    group_column = str(configuration['cross_validation'].get('group_column', 'image_id'))
    if group_column not in manifest.columns:
        real_training_ready = False
        readiness_message = f'Missing cross-validation group column: {group_column}'
    else:
        group_count = manifest.loc[trainable, group_column].str.strip().replace('', pd.NA).nunique()
        required_groups = int(configuration['cross_validation'].get('n_splits', 5))
        real_training_ready = group_count >= required_groups
        readiness_message = f'{group_count} valid groups; {required_groups} required'
else:
    train_count = int((trainable & (manifest['split'] == 'train')).sum())
    val_count = int((trainable & (manifest['split'] == 'val')).sum())
    real_training_ready = train_count > 0 and val_count > 0
    readiness_message = f'{train_count} train image(s), {val_count} validation image(s)'
print('Real training ready:', real_training_ready)
print('Reason:', readiness_message)
if not real_training_ready:
    print('This is expected for the current single test image or a pseudo-only manifest.')

## Inspect the model contract

This cell builds the configured U-Net and verifies an output tensor without training it. It is a quick architecture and installation check.

In [ ]:
import torch

from astroseg.models import build_model

model_config = configuration['model']
model = build_model(
    model_config['architecture'],
    int(model_config['input_channels']),
    int(model_config['num_classes']),
    int(model_config.get('base_channels', 32)),
)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
with torch.inference_mode():
    test_output = model(torch.zeros(1, 3, 64, 64))
print('Model:', model.__class__.__name__)
print('Parameters:', f'{parameter_count:,}')
print('Test input -> output:', (1, 3, 64, 64), '->', tuple(test_output.shape))
print('Available device:', 'cuda' if torch.cuda.is_available() else 'cpu')

## Run the synthetic overfit smoke test

Passing means loss decreased on one generated sample. It validates software plumbing only; it is not evidence of biological accuracy.

In [ ]:
if RUN_SMOKE_TEST:
    command = [
        sys.executable, 'scripts/train.py',
        '--smoke-test', '--smoke-steps', str(SMOKE_STEPS),
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Smoke test skipped because RUN_SMOKE_TEST=False')

## Optional real training

This cell refuses to train when the readiness audit fails. Pseudo labels are excluded because `configs/train_binary.yaml` lists only `seed`, `corrected`, and `reviewed` states.

In [ ]:
if RUN_REAL_TRAINING:
    if not real_training_ready:
        raise RuntimeError(f'Real training prerequisites are not satisfied: {readiness_message}')
    command = [sys.executable, 'scripts/train.py', '--config', str(RUNTIME_CONFIG_PATH)]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Real training disabled. Set RUN_REAL_TRAINING=True after the readiness audit passes.')

In [ ]:
import matplotlib.pyplot as plt

history_path = OUTPUT_DIRECTORY / 'history.csv'
if history_path.is_file():
    history = pd.read_csv(history_path)
    figure, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['epoch'], history['train_loss'], label='train')
    axes[0].plot(history['epoch'], history['validation_loss'], label='validation')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[1].plot(history['epoch'], history['train_dice'], label='train')
    axes[1].plot(history['epoch'], history['validation_dice'], label='validation')
    axes[1].set_title('Foreground Dice')
    axes[1].legend()
    figure.tight_layout()
    plt.show()
    history.tail()
else:
    print('No training history yet:', history_path)

## Optional checkpoint evaluation

Evaluation reconstructs complete images from overlapping patches before computing Dice, IoU, precision, and recall. Run it only on a split with human-validated annotations.

In [ ]:
checkpoint_path = OUTPUT_DIRECTORY / 'best.pt'
metrics_path = Path('outputs/metrics/notebook_evaluation.csv')
if RUN_EVALUATION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')
    annotated_in_split = int((trainable & (manifest['split'] == EVALUATION_SPLIT)).sum())
    if annotated_in_split == 0 and not cross_validation_enabled:
        raise RuntimeError(f'No trainable annotations in split {EVALUATION_SPLIT!r}')
    command = [
        sys.executable, 'scripts/evaluate.py',
        '--config', str(RUNTIME_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--split', EVALUATION_SPLIT,
        '--output', str(metrics_path),
    ]
    subprocess.run(command, check=True)
    evaluation = pd.read_csv(metrics_path)
else:
    evaluation = None
    print('Evaluation disabled. Enable it after training and validation data exist.')
evaluation

## Optional full-image prediction

Prediction does not require an astrocyte annotation, but it does require a trained checkpoint, prepared GFAP channel, nucleus labels, and at least one manifest row assigned to the requested split.

In [ ]:
prediction_directory = Path('outputs/predictions/notebook')
if RUN_PREDICTION:
    if not checkpoint_path.is_file():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')
    if not (manifest['split'] == PREDICTION_SPLIT).any():
        raise RuntimeError(f'No images assigned to split {PREDICTION_SPLIT!r}')
    command = [
        sys.executable, 'scripts/predict.py',
        '--config', str(RUNTIME_CONFIG_PATH),
        '--checkpoint', str(checkpoint_path),
        '--split', PREDICTION_SPLIT,
        '--output-dir', str(prediction_directory),
    ]
    subprocess.run(command, check=True)
else:
    print('Prediction disabled. Enable it after a trained checkpoint exists.')

In [ ]:
overlay_paths = sorted((prediction_directory / 'overlays').glob('*.png')) if prediction_directory.is_dir() else []
if overlay_paths:
    figure, axes = plt.subplots(1, min(3, len(overlay_paths)), figsize=(6 * min(3, len(overlay_paths)), 6), squeeze=False)
    for axis, path in zip(axes.flat, overlay_paths[:3]):
        axis.imshow(plt.imread(path))
        axis.set_title(path.stem)
        axis.axis('off')
    figure.tight_layout()
    plt.show()
else:
    print('No prediction overlays to display yet.')

## Interpretation and next steps

- A passing smoke test means the implementation works; it says nothing about research performance.
- Report validation metrics per independent image/well, not per randomly mixed patch.
- Keep test images untouched until the final evaluation.
- Inspect full-resolution overlays; scalar metrics can hide biologically important failure modes.
- Correct automatic proposals, update their state to `corrected` or `reviewed`, retrain, and repeat.

For the current single test image, the expected stopping point is a passing smoke test plus the automatic outputs from notebooks 01–03. Real training begins only after several independent human-validated images exist.